In [1]:
!pip install astroquery --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from astropy.time import Time
from astroquery.jplhorizons import Horizons

In [3]:
def mag_abs(mag, alpha, dist, r, G=0.15):
    """
    Calculate the absolute magnitude H using the asteroid H-G model.

    Parameters
    ----------
    mag : float or array-like
        Apparent magnitude.
    alpha : float or array-like
        Phase angle in degrees.
    dist : float or array-like
        Observer-object distance Delta in au.
    r : float or array-like
        Heliocentric distance in au.
    G : float
        Asteroid slope parameter (default 0.15).

    Returns
    -------
    H : float or array-like
        Absolute magnitude.
    """
    alpha = np.radians(alpha)

    phi1 = np.exp(-3.33 * np.tan(alpha / 2)**0.63)
    phi2 = np.exp(-1.87 * np.tan(alpha / 2)**1.22)

    phase = (1 - G) * phi1 + G * phi2

    H = mag - 5 * np.log10(r * dist) + 2.5 * np.log10(phase)

    return H

In [4]:
def query_dists_and_alpha(nombre_objeto, mjdobs):
    """
    Parameters
    ----------
    nombre_objeto : str
        Object designation recognized by JPL Horizons.
    mjdobs : float
        Observation time in Modified Julian Date.|

    Returns
    -------
    dist : float
        Observer-object distance in au.
    alpha : float
        Phase angle in degrees.
    r : float
        Heliocentric distance in au.
    """
    epoch_jd = Time(mjdobs, format='mjd', scale='utc').jd

    obj = Horizons(
        id=nombre_objeto,
        location='500',
        epochs=epoch_jd
    )

    eph = obj.ephemerides()

    dist = float(eph['delta'][0])
    alpha = float(eph['alpha'][0])
    r = float(eph['r'][0])

    return dist, alpha, r

In [5]:
def query_heliocentric_elements(nombre_objeto, mjdobs):
    """
    Parameters
    ----------
    nombre_objeto : str
        Object designation recognized by JPL Horizons.
    mjdobs : float
        Observation time in Modified Julian Date.

    Returns
    -------
    a : float
        Heliocentric semimajor axis in au.
    e : float
        Orbital eccentricity.
    i : float
        Heliocentric inclination in degrees.
    omega : float
        Argument of perihelion in degrees.
    Omega : float
        Longitude of ascending node in degrees.
    M : float
        Mean anomaly in degrees.
    """
    epoch_jd = Time(mjdobs, format='mjd', scale='utc').jd

    obj = Horizons(
        id=nombre_objeto,
        location='@sun',
        epochs=epoch_jd
    )

    elements = obj.elements()

    a = float(elements['a'][0])
    e = float(elements['e'][0])
    i = float(elements['incl'][0])
    omega = float(elements['w'][0])
    Omega = float(elements['Omega'][0])
    M = float(elements['M'][0])

    return a, e, i, omega, Omega, M

In [6]:
# this function calculates dist, alpha, r, a, e, i, omega, Omega, M

def fill_table(object_name, obs_time):

  dist, alpha, r = query_dists_and_alpha('2025 MY53', 60904.315457)

  a, e, i, omega, Omega, M = query_heliocentric_elements('2025 MY53', 60904.315457)

  return dist, alpha, r, a, e, i, omega, Omega, M

In [7]:
# load the data

df = pd.read_csv('/content/df_2016_GC22.csv', usecols=['provid', 'mjdobs', 'mag', 'band'])
df_2016_GC22 = df[df['provid']=='2016 GC22']
df_2016_GC22.shape

(769, 4)

In [8]:
object_names = df_2016_GC22['provid']
obs_times = df_2016_GC22['mjdobs']
mags = df_2016_GC22['mag']
bands = df_2016_GC22['band']

N = len(mags)

# example data
# object_names = ['2025 MY53', '2025 MY53', '2025 MY53']
# obs_times = [60904.315457, 60904.315457, 60904.315457]
# mags = [20,20.1,19.87]

# N = ...

In [9]:
# do the query and generate the master catalog

dist_ = []
alpha_ = []
r_ = []
a_ = []
e_ = []
i_ = []
omega_ = []
Omega_ = []
M_ = []

for index in range(N):

  print("Query number: "+str(index))

  dist, alpha, r, a, e, i, omega, Omega, M = fill_table(object_names.iloc[index], obs_times.iloc[index])

  dist_.append(dist)
  alpha_.append(alpha)
  r_.append(r)
  a_.append(a)
  e_.append(e)
  i_.append(i)
  omega_.append(omega)
  Omega_.append(Omega)
  M_.append(M)

df_master = pd.DataFrame({
    'object_names': object_names,
    'obs_times': obs_times,
    'mag': mags,
    'band': bands,
    'dist': dist_,
    'alpha': alpha_,
    'r': r_,
    'a': a_,
    'e': e_,
    'i': i_,
    'omega': omega_,
    'Omega': Omega_,
    'M': M_
})

Query number: 0
Query number: 1
Query number: 2
Query number: 3
Query number: 4
Query number: 5
Query number: 6
Query number: 7
Query number: 8
Query number: 9
Query number: 10
Query number: 11
Query number: 12
Query number: 13
Query number: 14
Query number: 15
Query number: 16
Query number: 17
Query number: 18
Query number: 19
Query number: 20
Query number: 21
Query number: 22
Query number: 23
Query number: 24
Query number: 25
Query number: 26
Query number: 27
Query number: 28
Query number: 29
Query number: 30
Query number: 31
Query number: 32
Query number: 33
Query number: 34
Query number: 35
Query number: 36
Query number: 37
Query number: 38
Query number: 39
Query number: 40
Query number: 41
Query number: 42
Query number: 43
Query number: 44
Query number: 45
Query number: 46
Query number: 47
Query number: 48
Query number: 49
Query number: 50
Query number: 51
Query number: 52
Query number: 53
Query number: 54
Query number: 55
Query number: 56
Query number: 57
Query number: 58
Query n

In [10]:
# add the H magnitudes to the master catalog

# calculate H
df_master['H'] = df_master.apply(
    lambda row: mag_abs(
        mag=row['mag'],
        alpha=row['alpha'],
        dist=row['dist'],
        r=row['r']
    ),
    axis=1
)

# divide in each band
bands_ = ['Li', 'Lg', 'Lz', 'Lr', 'Ly', 'Lu']
for band in bands_:
    df_master[f'H_{band}'] = df_master['H'].where(df_master['band'] == band)

In [11]:
# show the master catalog

df_master

,object_names,obs_times,mag,band,dist,alpha,r,a,e,i,omega,Omega,M,H,H_Li,H_Lg,H_Lz,H_Lr,H_Ly,H_Lu
0,2016 GC22,61096.129767,22.127,Li,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.46519,16.46519,NaN,NaN,NaN,NaN,NaN
1,2016 GC22,61096.128381,22.265,Li,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.60319,16.60319,NaN,NaN,NaN,NaN,NaN
2,2016 GC22,61096.126516,22.195,Li,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.53319,16.53319,NaN,NaN,NaN,NaN,NaN
3,2016 GC22,61096.131170,22.205,Li,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.54319,16.54319,NaN,NaN,NaN,NaN,NaN
4,2016 GC22,61096.130234,22.173,Li,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.51119,16.51119,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
764,2016 GC22,61095.119692,22.350,Lz,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.68819,NaN,NaN,16.68819,NaN,NaN,NaN
765,2016 GC22,61096.290594,22.516,Lz,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.85419,NaN,NaN,16.85419,NaN,NaN,NaN
766,2016 GC22,61097.301433,22.422,Lz,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.76019,NaN,NaN,16.76019,NaN,NaN,NaN
767,2016 GC22,61097.301903,22.419,Lz,3.180424,17.9605,2.774207,2.706548,0.026157,6.437071,339.955122,76.459032,162.449283,16.75719,NaN,NaN,16.75719,NaN,NaN,NaN
